# Algorítmo de Q-Learning

En este ejercicio vamos a probar el algorítmo de Q-learning como un representatnte de los métodos off-policy. Nuestro objetivo, es evaluar el algoritmo sobre distintos ambientes. Para cada uno de los ambientes deben ejecutar un agente de Q-learning en el ambiente, evaluar su ejecución y validar la efectividad del aprendizaje del agente entrenado sobre el ambiente.


El algoritmo de Q-learning está basado en la siguente funcionalidad (que puede estar integrado con el agente o ser implementado dentro de su propio modulo).

- El algoritmo debe tener acceso a todos los parámatros de aprendizaje (e.g., `alpha`, `gamma`, `epsilon`), el ambente, el agente y la memoria (la q-tabla).

Junto con las siguientes funciones:
- `run` que no recive ningún parámetro y se encarga de ejecutar el cíclo del agente (hasta llegar a convergencia) escogiendo la acción a ejecutar, ejecutando el paso y actualizando la función de aprendizaje
- `step` que recive una accióna ejecutar y la ejecuta efectivamente, retornando la recompensa, la informaciǿn si se llego al final del episodio y el estado de final luego de ejecutar la acción
- `get_reward` que recive la acción, el estado actual y el nuevo estado para calcular la recompensa que obtiene el agente de la ejecución de la acción.


## Task 1.  

Como primera tareas, utilice la implementación de Q-learning en el ambiente de cliff-walk (basado en el ambiente de Gridworld utilizdo anteriormente).
Recuerde que en este ambiente la recompensa por caer al barranco es de -100 y la recompensa de cada paso es -1.
Para la ejecución vamos a suponer acciones determinísticas.

![cliff-walk](img/cliff-walk.png)

Cuál es la diferencia entre las dos implementaciones (SARSA y Q-learning) del problema (que observaciones puede hacer)?

## Task 2. 

Para esta tarea vamos a utilizar un ambiente más complejo donde el agente debe completar tareas intermedias para completar el objetivo del agente.

![Locked-door](img/locked-door.png)

El ambiente consiste en dos habitaciones separadas por una puerta con llave. El objetivo del agente, que aparece en el cuarto de la izquierda, es llegar a la casilla de salida en el cuarto de la derecha (desconosida para el agente).

Para poder alcanzar el objetivo, el agente debe abrir la puerta con una llave (del mismo color de la puerta, que por defecto siempre lo será (azul)) que se encuentra en alguna posición en el cuarto. Sin embargo para poder abrir la puerta, el agente debe primero retirar una bola que se encuentra bloqueandola.

Por lo tanto, para poder alcanzar el objetivo, el agente tiene, ademas de las acciones de movimiento, acciones para recoger un objeto (llave o bola) y para abrir la puerta.

Su misión es definir el problema (sistema de recompensas) para permitir al agente alcanzar el objetivo. Entrenar al agente en el ambiente y poder explotarlo. Además de esto su programa debe ser capaz de grabar la memoria del agente (i.e., la Q-tabla).

Asegurese de guardar la información de las distintas pruebas que realice sobre el agente, para dejarlas ver en un pequeño reporte presentado los resultados.

## Task 3.

Como una tercera tarea adicional, vamos a probar (probablemente sin exito) la adaptabilidad del agente generado (i.e., el agente entrenado en el punto anterior).

Para esto pruebe los siguientes escenarios:
1. Aleatorize el punto de inicio del agente
1. Introduzca llaves de distintos colores en el ambiente, que se generar en posiciones aleatorias.

Entregue una solución para cada uno de los escenarios.


---
# Analisis: Q-Learning

**Tomas Acosta Bernal - 202011237**

Implementaciones:
- `q_learning.py` - Agente Q-Learning (con metodos `run`, `step`, `get_reward`)
- `cliff_walk_environment.py` - Ambiente Cliff-Walk
- `locked_door_environment.py` - Ambiente Locked-Door
- `locked_door_extended.py` - Ambiente Locked-Door extendido (para Task 3)

---
## Task 1: Q-Learning en Cliff-Walk

In [ ]:
from cliff_walk_environment import CliffWalk
from q_learning import QLearning
import numpy as np

# Entrenar Q-Learning en Cliff-Walk
np.random.seed(42)
env = CliffWalk()

print("Ambiente Cliff-Walk:")
env.print_grid()

agent = QLearning(env, alpha=0.81, gamma=0.96, epsilon=0.9, num_episodes=2000)
rewards = agent.run()

print("Politica Q-Learning:")
policy = agent.get_policy()
action_display = {'up': '  UP  ', 'down': ' DOWN ', 'left': ' LEFT ', 'right': 'RIGHT '}
for r in range(env.nrows):
    row_str = ""
    for c in range(env.ncols):
        state = (r, c)
        if state == env.goal:
            row_str += " GOAL "
        elif state in env.cliff:
            row_str += " XXXX "
        else:
            row_str += action_display.get(policy.get(state, '?'), '  ?   ')
    print(row_str)

print("\nTrayectoria greedy:")
path = agent.print_path()
print(f"Longitud: {len(path) - 1} pasos")
print(f"Recompensa promedio (ultimos 100): {np.mean(rewards[-100:]):.1f}")

### Comparacion SARSA vs Q-Learning en Cliff-Walk

| Aspecto | SARSA (on-policy) | Q-Learning (off-policy) |
|---------|-------------------|------------------------|
| **Actualizacion** | $Q(s,a) \leftarrow ... + \alpha[\gamma Q(s', a')]$ donde $a'$ es la accion que **realmente se toma** | $Q(s,a) \leftarrow ... + \alpha[\gamma \max_{a'} Q(s', a')]$ usa el **maximo** sobre acciones |
| **Camino aprendido** | Seguro, lejos del barranco (~23 pasos) | Optimo, al borde del barranco (~13 pasos) |
| **Riesgo** | Conservador - considera el riesgo de exploracion | Agresivo - aprende la politica optima independiente de la exploracion |

**Observaciones:**

1. **Q-Learning aprende el camino optimo** (fila 4, justo encima del barranco). Esto ocurre porque Q-Learning es off-policy: usa $\max Q(s', a')$ para actualizar, lo cual representa el valor de la mejor accion posible, no la accion que realmente se toma (que podria ser aleatoria por exploracion).

2. **SARSA aprende un camino seguro** (filas superiores, lejos del barranco). SARSA es on-policy: actualiza con $Q(s', a')$ donde $a'$ es la accion que realmente se elige. Como el agente explora con probabilidad $1-\epsilon$, hay riesgo de caer al barranco al estar cerca de el, y SARSA incorpora ese riesgo en sus Q-valores.

3. **Durante entrenamiento** (con exploracion), SARSA obtiene mejor recompensa promedio porque su politica es mas segura. Q-Learning obtiene peor recompensa durante entrenamiento porque su politica greedy lo lleva al borde del barranco donde la exploracion lo puede matar.

4. **En explotacion** (sin exploracion), Q-Learning es superior porque su camino es mas corto.

---
## Task 2: Locked-Door

### Diseno del ambiente

El ambiente Locked-Door consiste en una grilla de 5x11 con dos habitaciones separadas por una pared en la columna 5, con una puerta en la fila 2.

**Estado:** `(fila, columna, tiene_bola, tiene_llave, puerta_abierta)`

**Acciones:** `up, down, left, right, pick_up, open_door`

**Sistema de recompensas:**
| Evento | Recompensa |
|--------|-----------|
| Llegar al objetivo | +100 |
| Recoger bola | +10 |
| Recoger llave correcta | +10 |
| Abrir la puerta | +20 |
| Cada paso / accion invalida | -1 |

Las recompensas intermedias (+10, +20) guian al agente hacia los sub-objetivos necesarios.

In [ ]:
from locked_door_environment import LockedDoorEnv

# Mostrar el ambiente
env_ld = LockedDoorEnv()
print("Ambiente Locked-Door:")
print("A=Agente, K=Llave, B=Bola, D=Puerta, G=Meta, |=Pared")
env_ld.print_grid()
print(f"Espacio de estados: {len(env_ld.get_states())}")
print(f"Acciones: {env_ld.actions}")

In [ ]:
# Entrenar Q-Learning en Locked-Door
np.random.seed(42)
env_ld = LockedDoorEnv()
agent_ld = QLearning(env_ld, alpha=0.1, gamma=0.99, epsilon=0.9, num_episodes=5000)
rewards_ld = agent_ld.run()

print(f"Recompensa promedio (ultimos 100): {np.mean(rewards_ld[-100:]):.1f}")

# Guardar Q-tabla
agent_ld.save_q_table("q_table_locked_door.json")

In [ ]:
# Explotar el agente entrenado: ejecutar un episodio greedy
env_ld.reset()
agent_ld.epsilon = 1.0  # greedy
state = env_ld.get_current_state()
steps = 0

print("Ejecucion greedy del agente entrenado:")
print("=" * 60)
while not env_ld.is_terminal() and steps < 50:
    action = agent_ld.choose_action(state)
    reward, new_state = env_ld.do_action(action)
    r, c = new_state[0], new_state[1]
    has_ball, has_key, door_open = new_state[2], new_state[3], new_state[4]
    print(f"  Paso {steps:2d}: {action:10s} -> ({r},{c})  "
          f"bola={'SI' if has_ball else 'no'}  llave={'SI' if has_key else 'no'}  "
          f"puerta={'ABIERTA' if door_open else 'cerrada'}  R={reward:+d}")
    state = new_state
    steps += 1

print(f"\nObjetivo alcanzado en {steps} pasos")

### Resultados Task 2

El agente aprende exitosamente la secuencia correcta:
1. Navegar hasta la **bola** (2,4) y recogerla (`pick_up`)
2. Ir hasta la **llave** (4,3) y recogerla (`pick_up`)
3. Regresar junto a la **puerta** (2,4) y abrirla (`open_door`)
4. Cruzar a la habitacion derecha y navegar hasta el **objetivo** (4,10)

El sistema de recompensas intermedias (+10 por recoger, +20 por abrir puerta) es clave para guiar al agente. Sin estas recompensas, el agente tendria que descubrir una secuencia larga de acciones correctas antes de recibir cualquier senal positiva, haciendo el aprendizaje mucho mas lento.

La Q-tabla fue guardada en `q_table_locked_door.json` para poder cargarla y reutilizarla.

---
## Task 3: Pruebas de adaptabilidad

### Escenario 3.1: Punto de inicio aleatorio

In [ ]:
# Escenario 3.1: Probar el agente original (entrenado desde (0,0)) en posiciones aleatorias
print("=== Prueba: agente original en posiciones aleatorias ===")
print("(El agente fue entrenado SOLO desde la posicion (0,0))")
print()

test_starts = [(0, 0), (4, 4), (1, 2), (3, 0), (0, 4), (4, 0)]
for start in test_starts:
    env_ld.agent_start = start
    env_ld.reset()
    env_ld.current_state = (start[0], start[1], False, False, False)
    agent_ld.epsilon = 1.0
    state = env_ld.get_current_state()
    steps = 0
    while not env_ld.is_terminal() and steps < 50:
        action = agent_ld.choose_action(state)
        _, new_state = env_ld.do_action(action)
        state = new_state
        steps += 1
    reached = env_ld.is_terminal()
    print(f"  Inicio {start}: {'EXITO' if reached else 'FALLO'} en {steps} pasos")

In [ ]:
# Solucion 3.1: Entrenar con posiciones de inicio aleatorias
print("=== Solucion: entrenar con inicio aleatorio ===")
print()

np.random.seed(42)
left_room = [(r, c) for r in range(5) for c in range(5)]
env_rand = LockedDoorEnv()

agent_rand = QLearning(env_rand, alpha=0.1, gamma=0.99, epsilon=0.9, num_episodes=10000)

# Entrenamiento con inicio aleatorio
rewards_rand = []
for ep in range(10000):
    start = left_room[np.random.randint(len(left_room))]
    env_rand.agent_start = start
    env_rand.reset()
    
    state = env_rand.get_current_state()
    total_reward = 0
    steps = 0
    
    while not env_rand.is_terminal() and steps < 200:
        action = agent_rand.choose_action(state)
        reward, done, new_state = agent_rand.step(action)
        total_reward += reward
        
        actions_next = env_rand.get_possible_actions(new_state)
        max_q = max([agent_rand.Q.get((new_state, a), 0.0) for a in actions_next], default=0.0)
        old_q = agent_rand.Q.get((state, action), 0.0)
        agent_rand.Q[(state, action)] = (1 - agent_rand.alpha) * old_q + \
                                         agent_rand.alpha * (reward + agent_rand.gamma * max_q)
        state = new_state
        steps += 1
    
    rewards_rand.append(total_reward)
    if (ep + 1) % 2000 == 0:
        print(f"  Episodio {ep+1}/10000 - Avg reward (ult 100): {np.mean(rewards_rand[-100:]):.1f}")

# Evaluar
print("\nEvaluacion con inicio aleatorio:")
for start in test_starts:
    env_rand.agent_start = start
    env_rand.reset()
    env_rand.current_state = (start[0], start[1], False, False, False)
    agent_rand.epsilon = 1.0
    state = env_rand.get_current_state()
    steps = 0
    while not env_rand.is_terminal() and steps < 50:
        action = agent_rand.choose_action(state)
        _, new_state = env_rand.do_action(action)
        state = new_state
        steps += 1
    reached = env_rand.is_terminal()
    print(f"  Inicio {start}: {'EXITO' if reached else 'FALLO'} en {steps} pasos")

agent_rand.save_q_table("q_table_random_start.json")

### Escenario 3.2: Llaves de distintos colores en posiciones aleatorias

**Problema**: Si el agente fue entrenado con la llave en una posicion fija, no puede adaptarse a llaves en posiciones diferentes, porque esos estados nunca fueron visitados.

**Solucion**: Incluir la posicion de la llave en el estado del agente. Usamos `locked_door_extended.py` con estado: `(fila, col, tiene_bola, tiene_llave, puerta_abierta, llave_fila, llave_col)`.

Para llaves de color incorrecto: el agente no puede recogerlas (pick_up falla), lo cual es una limitacion inherente al problema.

In [ ]:
from locked_door_extended import LockedDoorExtended

# Definir posiciones posibles para la llave
key_positions = [(0, 0), (0, 4), (2, 0), (4, 3), (4, 0), (1, 2), (3, 1)]

np.random.seed(42)
env_ext = LockedDoorExtended(key_positions=key_positions, randomize_start=True)
print(f"Espacio de estados extendido: {len(env_ext.get_states())}")

agent_ext = QLearning(env_ext, alpha=0.1, gamma=0.99, epsilon=0.9, num_episodes=20000)
rewards_ext = agent_ext.run()

print(f"\nRecompensa promedio (ultimos 100): {np.mean(rewards_ext[-100:]):.1f}")
agent_ext.save_q_table("q_table_extended.json")

In [ ]:
# Evaluar el agente extendido con diferentes combinaciones
print("Evaluacion del agente extendido:")
print("=" * 60)

test_configs = [
    ((0, 0), (0, 0)),
    ((0, 0), (4, 3)),
    ((0, 0), (1, 2)),
    ((3, 2), (0, 0)),
    ((3, 2), (4, 3)),
    ((4, 4), (3, 1)),
]

for start, kp in test_configs:
    env_ext.agent_start = start
    env_ext.key_pos = kp
    env_ext.current_state = (start[0], start[1], False, False, False, kp[0], kp[1])
    agent_ext.epsilon = 1.0
    state = env_ext.get_current_state()
    steps = 0
    while not env_ext.is_terminal() and steps < 50:
        action = agent_ext.choose_action(state)
        _, new_state = env_ext.do_action(action)
        state = new_state
        steps += 1
    reached = env_ext.is_terminal()
    print(f"  Inicio {start}, Llave en {kp}: {'EXITO' if reached else 'FALLO'} en {steps} pasos")

In [ ]:
# Prueba con llave de color incorrecto
print("=== Prueba: llave de color incorrecto ===")
print("(La puerta es azul, la llave es roja)")
print()

np.random.seed(42)
env_wrong = LockedDoorEnv(key_color='red')
agent_wrong = QLearning(env_wrong, alpha=0.1, gamma=0.99, epsilon=0.9, num_episodes=3000)
rewards_wrong = agent_wrong.run()

print(f"\nRecompensa promedio (ultimos 100): {np.mean(rewards_wrong[-100:]):.1f}")
print("\nResultado: El agente NO puede completar la tarea porque la llave roja")
print("no puede abrir la puerta azul. El pick_up falla al intentar recoger")
print("la llave de color incorrecto, por lo que el agente nunca puede abrir la puerta.")

### Conclusiones Task 3

**Escenario 3.1 - Inicio aleatorio:**
- El agente original (entrenado desde una posicion fija) puede fallar desde posiciones no visitadas, porque no tiene Q-valores para esos estados.
- **Solucion:** Entrenar con posiciones de inicio aleatorias. Al explorar desde todas las posiciones del cuarto izquierdo, el agente aprende una politica general que funciona desde cualquier punto.

**Escenario 3.2 - Llaves de distintos colores en posiciones aleatorias:**
- **Llave de color incorrecto:** El agente no puede completar la tarea. Esto es una limitacion real: si la unica llave disponible es del color equivocado, el problema no tiene solucion.
- **Posiciones aleatorias:** El agente original no se adapta porque su estado `(fila, col, tiene_bola, tiene_llave, puerta_abierta)` no codifica donde esta la llave.
- **Solucion:** Extender el estado para incluir la posicion de la llave: `(fila, col, tiene_bola, tiene_llave, puerta_abierta, llave_fila, llave_col)`. Esto aumenta el espacio de estados pero permite al agente generalizar.

**Observacion general:** Q-Learning tabular tiene una limitacion fundamental: no puede generalizar a estados no visitados. Para ambientes con alta variabilidad, se necesitarian metodos de aproximacion de funciones (e.g., Deep Q-Learning) que puedan interpolar entre estados similares.